In [ ]:
#from goku.util.context import GokuContext
import sys
from datetime import datetime
from dateutil.relativedelta import relativedelta
import subprocess as sp
from io import StringIO
import pandas as pd

In [ ]:
month_value = "OLDEST_MONTH_VALUE_TO_SET"
date_24_month_ago = datetime.today().replace(day=1) - relativedelta(months=24)
month_to_query_from = date_24_month_ago.replace(
    hour=0, minute=0, second=0, microsecond=0
)
str_month_to_query_from = month_to_query_from.strftime("%Y-%m-%d")

In [ ]:
def fetch_data(file_name):
    swh_query = (
        open(f"{file_name}.sql")
        .read()
        .replace(month_value, f"'{str_month_to_query_from}'")
    )
    data = pd.read_csv(
        StringIO(
            sp.check_output(
                f"""pharos sql run --sql "{swh_query}" | jq -r '.result.data'""",
                shell=True,
            ).decode("utf-8")
        )
    )
    return data

In [ ]:
data_list = []
sql_files = [
    "metrics",
    "input_type_metrics",
    "selection_type_metrics",
    "validation_usages_metrics",
    "materialization_metrics",
]

for sql_file in sql_files:
    data_list.append(dict({"name": sql_file, "df": fetch_data(sql_file)}))

In [ ]:
for dt in data_list:
    print(f"{dt['name']}.csv")
    print(dt['df'].head(), end="\n\n")